# PersonaGym Experiment 1: Base Phi-3 vs LoRA on Kaggle

This notebook is designed for Kaggle GPU sessions.

What it does:
- loads the base model `microsoft/Phi-3-mini-4k-instruct`
- loads your LoRA adapter from a Kaggle input dataset
- reads a PersonaGym-style benchmark file or directory
- optionally limits the run to the first 50 personas
- generates responses for both base and LoRA models
- saves JSONL outputs you can feed into PersonaGym evaluation via `--saved_responses`

Before running:
1. Upload this notebook to Kaggle.
2. Add your LoRA adapter folder as a Kaggle dataset.
3. Add the PersonaGym benchmark files as a Kaggle dataset, or point `benchmark_path` to the correct location.
4. If Hugging Face access is required for Phi-3, add your token as a Kaggle secret named `HF_TOKEN`.


In [ ]:
%pip -q install -U transformers peft accelerate bitsandbytes datasets sentencepiece

In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import PeftModel


@dataclass
class Config:
    base_model_id: str = "microsoft/Phi-3-mini-4k-instruct"
    adapter_path: str = "/kaggle/input/phi3-rolebench-phase1-full/phi3_rolebench_phase1_full"
    benchmark_path: str = "/kaggle/input/personagym-benchmark"
    output_dir: str = "/kaggle/working/personagym_outputs"
    max_personas: int = 50
    persona_start: int = 0
    persona_end: Optional[int] = None
    tasks: List[str] = field(default_factory=lambda: [
        "action_justification",
        "expected_action",
        "linguistic_habits",
        "persona_consistency",
        "toxicity_control",
    ])
    max_input_length: int = 2048
    max_new_tokens: int = 256
    do_sample: bool = False
    temperature: float = 0.7
    top_p: float = 0.9
    seed: int = 42
    use_4bit: bool = True
    trust_remote_code: bool = False
    attn_implementation: str = "eager"
    resume_existing_outputs: bool = True
    flush_every: int = 1


def find_adapter_path(default_path: str) -> str:
    path = Path(default_path)
    if path.exists():
        return str(path)
    candidates = sorted(Path("/kaggle/input").rglob("adapter_config.json"))
    if candidates:
        return str(candidates[0].parent)
    return default_path


def find_benchmark_path(default_path: str) -> str:
    path = Path(default_path)
    if path.exists():
        return str(path)

    preferred_names = [
        "personagym",
        "benchmark-v1",
        "benchmark",
        "questions",
        "persona",
    ]
    for candidate in sorted(Path("/kaggle/input").rglob("*")):
        name = candidate.name.lower()
        if any(token in name for token in preferred_names):
            if candidate.is_dir() or candidate.suffix.lower() in {".json", ".jsonl"}:
                return str(candidate)
    return default_path


cfg = Config()
cfg.adapter_path = find_adapter_path(cfg.adapter_path)
cfg.benchmark_path = find_benchmark_path(cfg.benchmark_path)
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
set_seed(cfg.seed)

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

print(json.dumps(asdict(cfg), indent=2))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())


## Benchmark loader

The loader below is intentionally flexible because PersonaGym-style benchmark exports may use slightly different field names. It scans a file or directory, reads `.json` and `.jsonl`, infers the task where possible, and normalizes each item to a common schema.


In [ ]:
TASK_ALIASES = {
    "action_justification": ["action justification", "action_justification", "justification"],
    "expected_action": ["expected action", "expected_action"],
    "linguistic_habits": ["linguistic habits", "linguistic_habits", "style", "habit"],
    "persona_consistency": ["persona consistency", "persona_consistency", "consistency"],
    "toxicity_control": ["toxicity control", "toxicity_control", "toxicity"],
}


def first_present(obj: Dict[str, Any], keys: Iterable[str], default: Any = None) -> Any:
    for key in keys:
        if key in obj and obj[key] not in (None, ""):
            return obj[key]
    return default


def to_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        parts = [to_text(item) for item in value]
        return "\n".join(part for part in parts if part)
    if isinstance(value, dict):
        preferred = first_present(value, ["text", "content", "description", "persona", "name"])
        if preferred:
            return to_text(preferred)
        return json.dumps(value, ensure_ascii=False)
    return str(value).strip()


def messages_to_prompt(messages: Any) -> str:
    if not isinstance(messages, list):
        return ""
    chunks = []
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        role = msg.get("role", "user")
        content = to_text(first_present(msg, ["content", "text", "value"]))
        if content:
            chunks.append(f"{role.upper()}: {content}")
    return "\n".join(chunks).strip()


def canonicalize_task_name(raw_task: str) -> str:
    haystack = raw_task.lower().strip()
    for canonical, aliases in TASK_ALIASES.items():
        if any(alias in haystack for alias in aliases):
            return canonical
    return haystack.replace(" ", "_") if haystack else "unknown"


def infer_task(record: Dict[str, Any], source_path: Path) -> str:
    raw_task = to_text(first_present(record, ["task", "task_type", "category", "benchmark_task", "subset"]))
    if raw_task:
        return canonicalize_task_name(raw_task)
    return canonicalize_task_name(source_path.as_posix())


def persona_fields(record: Dict[str, Any]) -> Dict[str, str]:
    raw_persona = first_present(record, ["persona", "persona_text", "profile", "character", "role", "bio", "description"], {})
    persona_name = to_text(first_present(record, ["persona_id", "persona_name", "name", "id", "character_name", "role_name"]))
    persona_text = ""

    if isinstance(raw_persona, dict):
        if not persona_name:
            persona_name = to_text(first_present(raw_persona, ["persona_id", "persona_name", "name", "id"]))
        persona_text = to_text(first_present(raw_persona, ["persona", "text", "profile", "description", "bio"]))
        if not persona_text:
            persona_text = json.dumps(raw_persona, ensure_ascii=False)
    else:
        persona_text = to_text(raw_persona)

    if not persona_name:
        persona_name = persona_text.splitlines()[0][:80] if persona_text else "unknown_persona"

    return {
        "persona_id": persona_name,
        "persona_text": persona_text,
    }


def normalize_record(record: Dict[str, Any], source_path: Path, row_id: int) -> Optional[Dict[str, Any]]:
    prompt = to_text(first_present(record, ["prompt", "question", "query", "input", "instruction"]))
    if not prompt:
        prompt = messages_to_prompt(record.get("messages"))
    if not prompt:
        return None

    persona_info = persona_fields(record)
    task = infer_task(record, source_path)
    example_id = to_text(first_present(record, ["example_id", "id", "uid", "instance_id"])) or f"{source_path.stem}-{row_id}"

    return {
        "example_id": example_id,
        "persona_id": persona_info["persona_id"],
        "persona_text": persona_info["persona_text"],
        "task": task,
        "prompt": prompt,
        "source_file": str(source_path),
        "raw": record,
    }


def expand_personagym_benchmark_obj(obj: Dict[str, Any], path: Path) -> List[Dict[str, Any]]:
    persona_text = path.stem
    rows: List[Dict[str, Any]] = []
    for raw_task, prompts in obj.items():
        if not isinstance(prompts, list):
            continue
        task = canonicalize_task_name(str(raw_task))
        for question_idx, prompt in enumerate(prompts):
            prompt_text = to_text(prompt)
            if not prompt_text:
                continue
            rows.append({
                "example_id": f"{path.stem}-{task}-{question_idx}",
                "persona_id": persona_text,
                "persona_text": persona_text,
                "task": task,
                "prompt": prompt_text,
                "source_file": str(path),
                "raw": {
                    "task": raw_task,
                    "question_index": question_idx,
                    "prompt": prompt,
                },
            })
    return rows


def load_json_file(path: Path) -> List[Dict[str, Any]]:
    if path.suffix == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)

    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ["data", "examples", "items", "records", "benchmark"]:
            value = obj.get(key)
            if isinstance(value, list):
                return value
        expanded = expand_personagym_benchmark_obj(obj, path)
        if expanded:
            return expanded
    return []


def collect_benchmark_records(path_like: str) -> List[Dict[str, Any]]:
    path = Path(path_like)
    if not path.exists():
        raise FileNotFoundError(f"Benchmark path not found: {path}")

    if path.is_file():
        files = [path]
    else:
        files = sorted(
            p for p in path.rglob("*")
            if p.is_file() and p.suffix.lower() in {".json", ".jsonl"}
        )

    records: List[Dict[str, Any]] = []
    for file_path in files:
        try:
            raw_rows = load_json_file(file_path)
        except Exception as exc:
            print(f"Skipping {file_path} because it could not be read: {exc}")
            continue

        for row_id, raw_row in enumerate(raw_rows):
            if not isinstance(raw_row, dict):
                continue
            if {"example_id", "persona_id", "persona_text", "task", "prompt"}.issubset(raw_row.keys()):
                records.append(raw_row)
                continue
            normalized = normalize_record(raw_row, file_path, row_id)
            if normalized is not None:
                records.append(normalized)

    return records


def filter_records(
    records: List[Dict[str, Any]],
    tasks: List[str],
    max_personas: int,
    persona_start: int,
    persona_end: Optional[int],
) -> List[Dict[str, Any]]:
    allowed_tasks = {task.lower() for task in tasks}
    filtered = [row for row in records if row["task"] in allowed_tasks or row["task"] == "unknown"]

    seen_personas = []
    kept_personas = set()
    for row in filtered:
        key = row["persona_id"] or row["persona_text"] or "unknown_persona"
        if key not in kept_personas:
            kept_personas.add(key)
            seen_personas.append(key)

    persona_pool = seen_personas[:max_personas] if max_personas else seen_personas
    start = max(persona_start, 0)
    end = persona_end if persona_end is None else max(persona_end, start)
    selected_persona_list = persona_pool[start:end]
    selected_personas = set(selected_persona_list)
    print(
        f"Selected persona slice [{start}:{end if end is not None else 'end'}] "
        f"from {len(persona_pool)} persona candidates -> {len(selected_persona_list)} personas"
    )
    return [row for row in filtered if (row["persona_id"] or row["persona_text"] or "unknown_persona") in selected_personas]


records = collect_benchmark_records(cfg.benchmark_path)
records = filter_records(records, cfg.tasks, cfg.max_personas, cfg.persona_start, cfg.persona_end)

if not records:
    raise RuntimeError(
        "No benchmark records were found. Check cfg.benchmark_path and the field names in your benchmark files."
    )

task_counts = Counter(row["task"] for row in records)
persona_count = len({row["persona_id"] or row["persona_text"] for row in records})

print(f"Loaded {len(records)} benchmark items across {persona_count} personas")
print("Task counts:", dict(task_counts))
print("Example item:")
print(json.dumps({k: v for k, v in records[0].items() if k != "raw"}, indent=2, ensure_ascii=False))


## Model loading and generation helpers

The notebook uses 4-bit loading by default because that is the safest setup on Kaggle GPUs for Phi-3 mini plus a LoRA adapter.


In [ ]:
def pick_compute_dtype() -> torch.dtype:
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if torch.cuda.is_available():
        return torch.float16
    return torch.float32


def make_quant_config() -> Optional[BitsAndBytesConfig]:
    if not cfg.use_4bit or not torch.cuda.is_available():
        return None
    compute_dtype = pick_compute_dtype()
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )


def load_model_and_tokenizer(base_model_id: str, adapter_path: Optional[str] = None):
    tokenizer_source = adapter_path if adapter_path and Path(adapter_path).exists() else base_model_id
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source,
        trust_remote_code=cfg.trust_remote_code,
        token=hf_token,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model_kwargs = {
        "trust_remote_code": cfg.trust_remote_code,
        "device_map": "auto",
        "token": hf_token,
        "attn_implementation": cfg.attn_implementation,
    }
    quant_config = make_quant_config()
    if quant_config is not None:
        model_kwargs["quantization_config"] = quant_config
    model_kwargs["dtype"] = pick_compute_dtype()

    model = AutoModelForCausalLM.from_pretrained(base_model_id, **model_kwargs)
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return tokenizer, model


def build_messages(record: Dict[str, Any]) -> List[Dict[str, str]]:
    persona_text = record["persona_text"].strip()
    persona_id = record["persona_id"].strip()
    persona_block = persona_text if persona_text else persona_id
    system_prompt = (
        "You are role-playing the following persona. Stay consistent with the persona, "
        "answer naturally, and do not mention that you were given instructions unless asked.\n\n"
        f"Persona:\n{persona_block}"
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": record["prompt"]},
    ]


def render_prompt(tokenizer, record: Dict[str, Any]) -> str:
    messages = build_messages(record)
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    system_content = messages[0]["content"]
    user_content = messages[1]["content"]
    return (
        f"<|system|>\n{system_content}\n<|end|>\n"
        f"<|user|>\n{user_content}\n<|end|>\n"
        f"<|assistant|>\n"
    )


def generate_text(model, tokenizer, record: Dict[str, Any]) -> str:
    prompt_text = render_prompt(tokenizer, record)
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_input_length,
    )
    device = getattr(model, "device", None)
    if device is None:
        device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    generate_kwargs = {
        "max_new_tokens": cfg.max_new_tokens,
        "do_sample": cfg.do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if cfg.do_sample:
        generate_kwargs["temperature"] = cfg.temperature
        generate_kwargs["top_p"] = cfg.top_p

    with torch.inference_mode():
        output = model.generate(**inputs, **generate_kwargs)

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return re.sub(r"\s+", " ", text).strip()


def unload_model(model) -> None:
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Run generation for the base model and the LoRA adapter

Outputs are written to `/kaggle/working/personagym_outputs/`.


In [ ]:
def load_existing_rows(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Skipping incomplete JSONL line {line_number} in {path.name}")
    return rows


def chunk_label() -> str:
    end_label = cfg.persona_end if cfg.persona_end is not None else "end"
    return f"{cfg.persona_start}_{end_label}"


def run_generation(model_label: str, base_model_id: str, adapter_path: Optional[str] = None) -> Path:
    output_path = Path(cfg.output_dir) / f"{model_label}_responses_{chunk_label()}.jsonl"
    existing_rows = load_existing_rows(output_path) if cfg.resume_existing_outputs else []
    completed_ids = {row.get("example_id") for row in existing_rows if row.get("example_id")}
    pending_records = [record for record in records if record["example_id"] not in completed_ids]

    print(
        f"{model_label}: found {len(completed_ids)} completed responses, "
        f"{len(pending_records)} remaining"
    )
    if not pending_records:
        return output_path

    tokenizer, model = load_model_and_tokenizer(base_model_id, adapter_path=adapter_path)
    writes_since_flush = 0

    with output_path.open("a", encoding="utf-8") as f:
        for record in tqdm(pending_records, desc=f"Generating with {model_label}"):
            response = generate_text(model, tokenizer, record)
            row = {
                "model_label": model_label,
                "base_model_id": base_model_id,
                "adapter_path": adapter_path,
                "example_id": record["example_id"],
                "persona_id": record["persona_id"],
                "persona_text": record["persona_text"],
                "task": record["task"],
                "prompt": record["prompt"],
                "response": response,
                "source_file": record["source_file"],
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            writes_since_flush += 1
            if writes_since_flush >= max(cfg.flush_every, 1):
                f.flush()
                os.fsync(f.fileno())
                writes_since_flush = 0

        if writes_since_flush:
            f.flush()
            os.fsync(f.fileno())

    unload_model(model)
    return output_path


base_output_path = run_generation("base", cfg.base_model_id)
lora_output_path = run_generation("lora", cfg.base_model_id, adapter_path=cfg.adapter_path)

print("Saved base responses to:", base_output_path)
print("Saved LoRA responses to:", lora_output_path)


## Build a side-by-side comparison file

This is useful for quick inspection before running PersonaGym evaluation.


In [ ]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    return load_existing_rows(path)


base_rows = read_jsonl(base_output_path)
lora_rows = read_jsonl(lora_output_path)

comparison_path = Path(cfg.output_dir) / f"base_vs_lora_comparison_{chunk_label()}.jsonl"
with comparison_path.open("w", encoding="utf-8") as f:
    for base_row, lora_row in zip(base_rows, lora_rows):
        merged = {
            "example_id": base_row["example_id"],
            "persona_id": base_row["persona_id"],
            "task": base_row["task"],
            "prompt": base_row["prompt"],
            "base_response": base_row["response"],
            "lora_response": lora_row["response"],
        }
        f.write(json.dumps(merged, ensure_ascii=False) + "\n")

print("Side-by-side comparison file:", comparison_path)
print("Preview:")
print(json.dumps(read_jsonl(comparison_path)[0], indent=2, ensure_ascii=False))


## Optional PersonaGym evaluation step

If you also upload or clone the PersonaGym evaluation code on Kaggle, point the command below to the correct evaluation script.

The important part is that PersonaGym supports a `--saved_responses` style input, so you can evaluate the already-generated JSONL files instead of forcing PersonaGym to call your Hugging Face models directly.


In [ ]:
# Example only. Update the path to match the PersonaGym repo layout you upload to Kaggle.
# !python /kaggle/input/personagym-repo/path/to/evaluate.py --saved_responses {base_output_path}
# !python /kaggle/input/personagym-repo/path/to/evaluate.py --saved_responses {lora_output_path}

In [ ]:
!ls -R /kaggle/working/personagym_outputs